# Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import einsum, rearrange, reduce, repeat

# GQA

In [2]:
# In PyTorch (4, 6, 2) @ (2, 2, 6): Fails - since broadcasting cannot happen
try:
    (torch.rand(4, 6, 2) @ torch.rand(2, 2, 6)).shape
except Exception as e:
    print(e)

The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 0


In [3]:
# In EinSUM, belwo code looks correct shape wise but is very wrong: 
# For each q matrix, we are doing matmul over all kv matrices
q = torch.randn(2, 4, 10, 4)
k = torch.randn(2, 2, 10, 4)
print(q.shape, k.shape)
dummy_attn = einsum(q, k, "b h i d, b g j d -> b h i j")
print(dummy_attn.shape)

torch.Size([2, 4, 10, 4]) torch.Size([2, 2, 10, 4])
torch.Size([2, 4, 10, 10])


In [4]:
q = torch.randn(2, 4, 10, 4)
print('q.shape:', q.shape)
k = torch.randn(2, 2, 10, 4)
print('k.shape:', k.shape)
k = repeat(k, "b g s d -> b (g r) s d", r=2)
print('k.shape:', k.shape)
print(q.shape, k.shape)
dummy_attn = einsum(q, k, "b h i d, b h j d -> b h i j")
print(dummy_attn.shape)

q.shape: torch.Size([2, 4, 10, 4])
k.shape: torch.Size([2, 2, 10, 4])
k.shape: torch.Size([2, 4, 10, 4])
torch.Size([2, 4, 10, 4]) torch.Size([2, 4, 10, 4])
torch.Size([2, 4, 10, 10])


In [5]:
# Mask by Addition
torch.triu(torch.full_like(dummy_attn[0, 0], -torch.inf), diagonal=1)

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [6]:
# Mask by masked_fill
mask = torch.tril(torch.ones_like(dummy_attn[0, 0]))==0
print(torch.randn(10, 10).masked_fill(mask, value=-torch.inf))

tensor([[-0.1467,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [ 0.0085, -1.7825,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [ 1.7144, -0.3760, -1.1813,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [-1.2845, -0.8422, -0.0811,  0.0381,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [-1.1002, -1.0134, -0.1263,  1.9501, -0.1490,    -inf,    -inf,    -inf,
            -inf,    -inf],
        [-1.4496,  0.6230, -0.6372,  0.2282,  1.4730, -0.4128,    -inf,    -inf,
            -inf,    -inf],
        [ 0.4642,  0.4170,  0.1281,  0.0272,  0.6580, -0.0554,  0.2725,    -inf,
            -inf,    -inf],
        [ 1.1667, -0.8978,  0.2694, -0.9393,  2.1950, -0.4879,  0.1762, -0.9991,
            -inf,    -inf],
        [-0.4747, -0.0331, -1.4389, -0.1847,  1.4540, -0.3924, -0.7112,  0.0729,
         -0.9153,    -inf],
        [-1.2505, -

In [7]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model:int, n_heads:int, n_groups:int):
        super().__init__()
        self.d_model=d_model
        self.n_heads=n_heads
        self.n_groups=n_groups
        self.q_dim = d_model//n_heads
        # QKV projection layer
        self.proj_qkv = nn.Linear(d_model, d_model + 2*self.n_groups*self.q_dim)
        # Output projection layer
        self.proj_out = nn.Linear(d_model, d_model)
        self.proj_out.RESIDUAL_PATH_SCALE_INIT=1
        # Dropouts
        self.dropout_attn = nn.Dropout(0.0)
        self.dropout_out = nn.Dropout(0.0)
        self.dropout_p = 0.0
        # Flash Attn
        self.use_flash = False
        # Debug Mode
        self.attn_debug=False


    def forward(self, x:torch.Tensor) -> torch.Tensor:
        # x.shape = (B, T, d_model)
        B, T, _ = x.shape

        # Get qkv projections in concated form
        qkv = self.proj_qkv(x)
        q, k, v = torch.split(qkv, [self.d_model, self.n_groups*self.q_dim, self.n_groups*self.q_dim], dim=-1)

        # Split q, k, v
        q = rearrange(q, "b s (h d) -> b h s d", h=self.n_heads)
        k = rearrange(k, "b s (g d) -> b g s d", g=self.n_groups)
        v = rearrange(v, "b s (g d) -> b g s d", g=self.n_groups)

        # Repeat k v to be shared across q heads
        k = repeat(k, "b g s d -> b (n g) s d", n=self.n_heads//self.n_groups)
        v = repeat(v, "b g s d -> b (n g) s d", n=self.n_heads//self.n_groups)
        
        # Attention
        if self.use_flash:
            # Implement Flash Attention 
            y = F.scaled_dot_product_attention(
                    q, k, v, 
                    attn_mask=None, 
                    dropout_p=self.dropout_p if self.training else 0, 
                    is_causal=True
                )
        else:
            # Implement Manual Attention 
            # Attention_p1: Dot Product (Q @ K.T)
            attn_scores = einsum(q, k, "b h i d, b h j d -> b h i j")
            # Attention_p2: Scaling
            head_dim = self.d_model//self.n_heads
            attn_scores *= (1/(head_dim)**0.5)
            # Attention_p3: Causal Mask-Way1
            mask = torch.triu(torch.full_like(attn_scores, fill_value=-torch.inf, device=x.device), diagonal=1)
            attn_scores += mask
            # # Attention_p3: Causal Mask-Way2
            # mask = torch.tril(torch.ones(T, T, device=x.device))==0
            # attn_scores = attn_scores.masked_fill(mask, value=-torch.inf)
            # Attention_p4: Softmax
            attn_scores = F.softmax(attn_scores, dim=-1)
            if self.attn_debug:
                self.attn_debug_probs = attn_scores.detach()
            attn_scores = self.dropout_attn(attn_scores)
            # Attention_p5: Dot Product (A @ V)
            y = einsum(attn_scores, v, "b h i j, b h j d -> b h i d")

        
        # Reshape post attention output to original shape of input
        y = rearrange(y, "b h s d -> b s (h d)")

        # Get op projection
        y = self.dropout_out(self.proj_out(y))


        return y


gqa = GroupedQueryAttention(16, 4, 2)
x = torch.randn(2, 10, 16)
print(x.shape)
y = gqa(x)
print(y.shape)

torch.Size([2, 10, 16])
torch.Size([2, 10, 16])


# KV Cache

In [8]:
class KVCache:
    def __init__(self, d_model:int, n_heads:int, n_kv:int, max_new_tokens:int):

        self.n_heads = n_heads
        self.n_kv = n_kv
        self.head_dim = d_model//n_heads
        self.max_new_tokens=max_new_tokens

        # Define tensors to store k and v vectors of past tokens
        self.k_cache = None
        self.v_cache = None

        # Current token pos
        self.curr_idx = 0

    def _prefill_cache(self, k:torch.Tensor, v:torch.Tensor):

        # k.shape = (B, T, n_kv*head_dim)
        # v.shape = (B, T, n_kv*head_dim)
        B, T, d_kv = k.shape

        # Check for kv dims
        assert d_kv==(self.n_kv*self.head_dim), (
            f"Mismatch in hidden dim for KV vectors (concated across all heads)",
            f"Dim Expected: {self.n_kv*self.head_dim}, Dim Got: {d_kv}"
        )

        # Define cache vectors of correct shape (prefilled with 0s) 
        self.k_cache = torch.zeros(B, (T+self.max_new_tokens), d_kv, device=k.device)
        self.v_cache = torch.zeros(B, (T+self.max_new_tokens), d_kv, device=v.device)

        # Update the KV values corresponding to prompt
        self.k_cache[:, :T] = k
        self.v_cache[:, :T] = v

        # Update pointer for next token position
        self.curr_idx = T

    def update_cache(self, k:torch.Tensor, v:torch.Tensor):

        if self.k_cache is None:
            # Prefill
            self._prefill_cache(k, v)
        else:
            # Decode
            # k.shape = (B, 1, n_kv*head_dim)
            # v.shape = (B, 1, n_kv*head_dim)
            B, T, d_kv = k.shape

            # Check for kv dims
            assert d_kv==(self.n_kv*self.head_dim), (
                f"Mismatch in hidden dim for KV vectors (concated across all heads)",
                f"Dim Expected: {self.n_kv*self.head_dim}, Dim Got: {d_kv}"
            )

            # Check for T (should be 1)
            assert T==1, (
                f"Expected 1 new token to increase KV cache length by, got {T}"
            )

            # Update the KV values corresponding to prompt
            self.k_cache[:, self.curr_idx:self.curr_idx+1] = k
            self.v_cache[:, self.curr_idx:self.curr_idx+1] = v

            # Update pointer for next token position
            self.curr_idx += 1


    def reset_cache(self):
        self.k_cache = None
        self.v_cache = None



kv_cache = KVCache(32, 4, 2, 6)

In [9]:
# Simulate Prefill
k = torch.randn(2, 10, 16)
v = torch.randn(2, 10, 16)


# Update KV Cache
kv_cache.update_cache(k, v)

print(kv_cache.curr_idx)
print(kv_cache.k_cache.shape)
print(kv_cache.k_cache[0, kv_cache.curr_idx-1])
print(kv_cache.k_cache[0, kv_cache.curr_idx])

10
torch.Size([2, 16, 16])
tensor([-1.1165, -0.1133,  0.3707, -1.9048, -0.6361, -0.3772, -0.1537, -1.0450,
        -1.3283, -0.9365, -0.7386,  0.2860, -0.6777, -0.4200, -0.2952,  0.1011])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [10]:
# Simulate Decode
for _ in range(6):

    k = torch.randn(2, 1, 16)
    v = torch.randn(2, 1, 16)

    # Update KV Cache
    kv_cache.update_cache(k, v)

    print(kv_cache.curr_idx)
    print(kv_cache.k_cache.shape)
    print(kv_cache.k_cache[0, kv_cache.curr_idx-2])
    print(kv_cache.k_cache[0, kv_cache.curr_idx-1])
    print('-'*50)

11
torch.Size([2, 16, 16])
tensor([-1.1165, -0.1133,  0.3707, -1.9048, -0.6361, -0.3772, -0.1537, -1.0450,
        -1.3283, -0.9365, -0.7386,  0.2860, -0.6777, -0.4200, -0.2952,  0.1011])
tensor([ 0.7090,  1.6289, -0.0250, -0.8235, -0.2635,  1.3422,  0.5456, -0.4178,
        -0.8203,  0.5403,  0.0527,  0.8886, -1.0780,  0.3825,  1.7729,  0.0067])
--------------------------------------------------
12
torch.Size([2, 16, 16])
tensor([ 0.7090,  1.6289, -0.0250, -0.8235, -0.2635,  1.3422,  0.5456, -0.4178,
        -0.8203,  0.5403,  0.0527,  0.8886, -1.0780,  0.3825,  1.7729,  0.0067])
tensor([-0.1625,  0.1235,  1.6792,  0.4964, -1.1574,  0.9218,  1.9919, -2.0769,
         1.0009, -0.3373, -0.6968,  0.2153, -1.0183, -1.9866, -3.2761, -2.1216])
--------------------------------------------------
13
torch.Size([2, 16, 16])
tensor([-0.1625,  0.1235,  1.6792,  0.4964, -1.1574,  0.9218,  1.9919, -2.0769,
         1.0009, -0.3373, -0.6968,  0.2153, -1.0183, -1.9866, -3.2761, -2.1216])
tensor([-0.0

In [11]:
kv_cache.k_cache[0]

tensor([[ 0.5384,  0.5748, -1.6683,  0.6254,  0.9343,  0.1257,  0.5392, -0.1478,
          1.3077,  0.9354, -0.8231,  0.4029, -1.4131,  0.9686, -0.5444,  2.4267],
        [-0.2749,  1.2427, -0.1701, -2.4855, -1.3618,  0.9083, -0.0557, -0.8659,
          0.6263, -1.4576, -0.4636,  0.6046,  0.5524,  0.1546, -0.6933, -0.6157],
        [ 0.4943, -1.6554, -0.5335, -0.2935,  0.3188,  1.2464, -0.6369, -0.3303,
          1.1853, -0.1198, -0.5262,  1.2057, -0.3292, -0.5051, -0.1945, -0.8256],
        [ 0.0845,  0.4215,  1.6692,  0.9326, -0.9595,  0.3119, -0.1848, -0.1174,
          0.9035,  0.0988,  0.7548,  0.2101,  0.2621, -0.4948,  1.1480,  1.1011],
        [ 1.5059,  0.7698,  0.9462,  0.4890,  0.7305, -0.0113,  0.4670,  2.0540,
         -0.1361,  0.4090,  0.9296, -0.8384,  1.3052, -0.7979,  1.1057, -0.0723],
        [-0.6284,  1.1868,  1.1025, -0.0824,  0.3164,  1.3299, -2.2221,  1.2634,
         -2.6005, -0.7202,  0.8248, -0.9771, -2.1802,  0.5053,  0.1170, -0.8871],
        [-2.1423,  0.7